# SIM Intervention Demand Forecasting

**YCIT 440 – AI and Data Science Capstone Project**

**Team 4**

## 1. Modeling Dataset: Data Preparation

### 1.1. Project Objective

The objective of this project is to forecast the total number of recorded interventions for each of the six operational divisions of the Service de sécurité incendie de Montréal (SIM), one calendar day ahead.

The forecasting models will use historical intervention counts available through D−3 to predict intervention demand for day D, with the forecast prepared on D−1.

The initial modeling approach will combine LightGBM, CatBoost, and Exponential Smoothing. Model performance will be evaluated against a baseline based on recent observations for the same day of the week, using Mean Absolute Error (MAE) as the primary metric.

### 1.2. Source Data and Modeling Dataset Structure

The original dataset contains individual SIM intervention records. These records will be processed and aggregated into daily intervention counts for each operational division.

Each observation in the modeling dataset will represent one division on one calendar day.

### 1.3. Import Required Libraries

We will use pandas to load, inspect, and prepare the intervention data. NumPy will support numerical operations during data preparation and modeling.

In [ ]:

import pandas as pd
import numpy as np


### 1.4. Load Original Intervention Datasets

The SIM intervention data is provided in historical and recent datasets.

Both datasets will be loaded separately before checking their structure, coverage, and compatibility. This will allow us to identify any potential overlap or inconsistencies before combining the records.

In [ ]:


# Original SIM dataset filenames
historical_file = "../../data/raw/donneesouvertes-interventions-sim2020.csv"
recent_file = "../../data/raw/donneesouvertes-interventions-sim.csv"

# Load the original datasets
df_historical = pd.read_csv(historical_file)
df_recent = pd.read_csv(recent_file)

# Display dataset dimensions
print("Historical dataset:", df_historical.shape)
print("Recent dataset:", df_recent.shape)
print("Total records:", len(df_historical) + len(df_recent))



Historical dataset: (539203, 13)
Recent dataset: (215210, 13)
Total records: 754413


### 1.5. Verify Dataset Structure

Before combining the datasets, we will compare their column names and data types to confirm that they have a compatible structure.

This verification is important because both datasets must use consistent fields for the modeling process.

In [ ]:

# Display column names and data types for the historical dataset
print("Historical Dataset:")
print(df_historical.dtypes)

print("\nRecent Dataset:")
print(df_recent.dtypes)

# Check whether both datasets have the same columns
same_columns = list(df_historical.columns) == list(df_recent.columns)

print("\nSame columns in the same order:", same_columns)


Historical Dataset:
INCIDENT_NBR            int64
CREATION_DATE_TIME        str
INCIDENT_TYPE_DESC        str
DESCRIPTION_GROUPE        str
CASERNE                 int64
NOM_VILLE                 str
NOM_ARROND                str
DIVISION                int64
NOMBRE_UNITES         float64
MTM8_X                float64
MTM8_Y                float64
LONGITUDE             float64
LATITUDE              float64
dtype: object

Recent Dataset:
INCIDENT_NBR            int64
CREATION_DATE_TIME        str
INCIDENT_TYPE_DESC        str
DESCRIPTION_GROUPE        str
CASERNE                 int64
NOM_VILLE                 str
NOM_ARROND                str
DIVISION                int64
NOMBRE_UNITES         float64
MTM8_X                float64
MTM8_Y                float64
LONGITUDE             float64
LATITUDE              float64
dtype: object

Same columns in the same order: True


### 1.6. Combine Historical and Recent Intervention Records

Both datasets have the same column names and data types.

We will combine them into a single DataFrame containing all available intervention records.

The original records will be preserved without modifications at this stage. We will check the combined dataset for potential overlaps and data quality issues before creating the daily intervention counts.

In [ ]:

# Combine the historical and recent datasets
df = pd.concat([df_historical, df_recent], ignore_index=True)

# Display the dimensions of the combined dataset
print("Combined dataset:", df.shape)

# Display the first five records
display(df.head())


Combined dataset: (754413, 13)


,INCIDENT_NBR,CREATION_DATE_TIME,INCIDENT_TYPE_DESC,DESCRIPTION_GROUPE,CASERNE,NOM_VILLE,NOM_ARROND,DIVISION,NOMBRE_UNITES,MTM8_X,MTM8_Y,LONGITUDE,LATITUDE
0,4579,2022-01-16T09:45:45,Feu / 4e Alerte,INCENDIE,45,Montréal,Mercier-Hochelaga-Maisonneuve,5,80.0,301939.7,5046439.7,-73.536637,45.557863
1,17503,2022-03-01T14:24:44,Premier répondant,1-REPOND,34,Montréal,Côte-des-Neiges-Notre-Dame-de-Grâce,3,1.0,294702.0,5037802.7,-73.629167,45.480077
2,24768,2022-03-29T09:42:01,Premier répondant,1-REPOND,53,Beaconsfield,Indéterminé,1,1.0,276414.2,5032320.4,-73.862773,45.430241
3,54618,2022-06-27T14:21:44,Premier répondant,1-REPOND,32,Montréal,Rivière-des-Prairies-Pointe-aux-Trembles,4,1.0,305738.5,5061616.6,-73.487949,45.694433
4,55101,2022-06-28T20:35:07,Premier répondant,1-REPOND,45,Montréal,Mercier-Hochelaga-Maisonneuve,5,1.0,301675.8,5046174.7,-73.540017,45.555478


### 1.7. Check Duplicate Records

Before aggregating the intervention records into daily counts, we will check for duplicate records.

We will examine both complete duplicate rows and repeated `INCIDENT_NBR` values.

Repeated identifiers require further investigation because they may affect how interventions are counted. No records will be removed until the results have been reviewed.

In [ ]:

# Count completely duplicated rows
duplicate_rows = df.duplicated().sum()

# Count repeated incident identifiers
duplicate_incident_ids = df["INCIDENT_NBR"].duplicated().sum()

# Count unique incident identifiers
unique_incidents = df["INCIDENT_NBR"].nunique()

print("Duplicate rows:", duplicate_rows)
print("Repeated incident IDs:", duplicate_incident_ids)
print("Unique incident IDs:", unique_incidents)


Duplicate rows: 0
Repeated incident IDs: 614582
Unique incident IDs: 139831


### 1.8. Investigate Repeated Incident Identifiers

The dataset contains no completely duplicated rows, but many `INCIDENT_NBR` values appear more than once.

We will examine examples of repeated identifiers and compare their dates and divisions to better understand how incident numbers are used.

At this stage, repeated identifiers will not be treated as duplicate interventions.

In [ ]:

# Find incident IDs that appear more than once
incident_counts = df["INCIDENT_NBR"].value_counts()

repeated_ids = incident_counts[incident_counts > 1]

# Display the five most frequently repeated IDs
print("Most frequently repeated incident IDs:")
print(repeated_ids.head(5))

# Examine records for the most frequently repeated ID
example_id = repeated_ids.index[0]

print("\nRecords for incident ID:", example_id)

display(
    df[df["INCIDENT_NBR"] == example_id][
        ["INCIDENT_NBR", "CREATION_DATE_TIME", "DIVISION", "CASERNE"]
    ].head(15)
)


Most frequently repeated incident IDs:
INCIDENT_NBR
4579     7
17503    7
24768    7
54618    7
55101    7
Name: count, dtype: int64

Records for incident ID: 4579


,INCIDENT_NBR,CREATION_DATE_TIME,DIVISION,CASERNE
0,4579,2022-01-16T09:45:45,5,45
47289,4579,2020-01-15T10:59:15,3,4
89890,4579,2023-01-17T06:55:53,3,42
148033,4579,2021-01-18T09:54:05,4,21
217797,4579,2024-01-15T10:44:12,2,64
668165,4579,2026-01-14T20:48:25,3,4
748804,4579,2025-01-13T14:55:35,3,37


### 1.9. Verify Incident Identifier Uniqueness Within Each Year

The investigation suggests that incident identifiers are reused across different years.

We will check whether the combination of year and `INCIDENT_NBR` uniquely identifies each record.

This verification will help determine whether repeated incident identifiers represent normal yearly numbering or require further investigation.

In [ ]:

# Extract the year from the original date string
df["Year"] = df["CREATION_DATE_TIME"].str[:4]

# Check repeated incident IDs within the same year
duplicate_year_incidents = df.duplicated(
    subset=["Year", "INCIDENT_NBR"]
).sum()

# Count unique combinations of year and incident ID
unique_year_incidents = df[
    ["Year", "INCIDENT_NBR"]
].drop_duplicates().shape[0]

print("Repeated incident IDs within the same year:", duplicate_year_incidents)
print("Unique year-incident combinations:", unique_year_incidents)
print("Total records:", len(df))


Repeated incident IDs within the same year: 0
Unique year-incident combinations: 754413
Total records: 754413


**Methodological Observation**

The combined dataset contains 754,413 records with no completely duplicated rows. Although 614,582 records have an `INCIDENT_NBR` value that appeared previously, the investigation showed that incident identifiers are reused across different years.

The combination of year and `INCIDENT_NBR` is unique for all 754,413 records. Therefore, no records were removed based on repeated incident identifiers, preserving the original intervention counts for modeling.

### 1.10. Validate Intervention Dates

The `CREATION_DATE_TIME` column contains the date and time associated with each intervention.

We will convert this column to datetime format and check for invalid or missing values. We will also identify the earliest and latest intervention dates to verify the temporal coverage of the combined dataset.

Records with invalid dates will be investigated before preparing the daily intervention counts.

In [ ]:

# Convert intervention timestamps to datetime
df["CREATION_DATE_TIME"] = pd.to_datetime(
    df["CREATION_DATE_TIME"],
    format="ISO8601",
    errors="coerce"
)

# Count records with invalid or missing dates
invalid_dates = df["CREATION_DATE_TIME"].isna().sum()

# Find the earliest and latest intervention dates
first_date = df["CREATION_DATE_TIME"].min()
last_date = df["CREATION_DATE_TIME"].max()

print("Invalid or missing dates:", invalid_dates)
print("First intervention date:", first_date)
print("Last intervention date:", last_date)


Invalid or missing dates: 0
First intervention date: 2020-01-01 00:01:37
Last intervention date: 2026-09-24 23:59:29


**Methodological Observation**

All 754,413 intervention records were successfully converted to datetime format, with no invalid or missing dates. The dataset covers January 1, 2020, through September 24, 2026, consistent with the revised Business Problem Framing Report.

However, valid timestamps do not guarantee complete daily coverage. A separate check is required to identify missing calendar days.

### 1.11. Identify Missing Calendar Days

We will compare the complete calendar date range with the dates present in the intervention records.

This check will identify any calendar days without published intervention records, including the previously identified gap on December 31, 2024.

Missing calendar days will not be interpreted as days with zero interventions.

In [ ]:

# Extract calendar dates from intervention timestamps
df["Date"] = df["CREATION_DATE_TIME"].dt.normalize()

# Create the complete calendar date range
all_dates = pd.date_range(
    start=df["Date"].min(),
    end=df["Date"].max(),
    freq="D"
)

# Identify dates present in the dataset
observed_dates = pd.DatetimeIndex(df["Date"].unique())

# Find missing calendar days
missing_dates = all_dates.difference(observed_dates)

print("Total calendar days:", len(all_dates))
print("Observed calendar days:", len(observed_dates))
print("Missing calendar days:", len(missing_dates))

print("\nMissing dates:")
print(missing_dates)


Total calendar days: 2459
Observed calendar days: 2458
Missing calendar days: 1

Missing dates:
DatetimeIndex(['2024-12-31'], dtype='datetime64[us]', freq='D')


**Methodological Observation**

The dataset contains records for 2,458 of the 2,459 calendar days between January 1, 2020, and September 24, 2026. December 31, 2024, is the only missing calendar day.

This finding confirms the data gap previously identified during the Exploratory Data Analysis (EDA).

The missing date will not be treated as a day with zero interventions because the absence of published records does not establish that no interventions occurred. The gap will remain missing in the modeling dataset, and no forecast will be scored against an unknown actual count for that date.

### 1.12. Verify Operational Divisions

The forecasting target includes SIM operational Divisions 1 through 6.

We will examine the number of intervention records associated with each division code and identify records outside the target divisions.

This verification will help determine which records should be included when constructing the daily intervention counts for modeling.

In [ ]:

# Count intervention records by division
division_counts = df["DIVISION"].value_counts().sort_index()

print("Intervention records by division:")
print(division_counts)

# Identify records outside the target divisions
outside_target = ~df["DIVISION"].isin([1, 2, 3, 4, 5, 6])

print("\nRecords outside Divisions 1-6:", outside_target.sum())


Intervention records by division:
DIVISION
0       600
1     69616
2    133278
3    150400
4    145196
5    142176
6    113147
Name: count, dtype: int64

Records outside Divisions 1-6: 600


**Methodological Observation**

The dataset contains intervention records for Divisions 0 through 6. Of the 754,413 records, 600 are assigned to Division 0.

Since the forecasting target is explicitly defined for operational Divisions 1 through 6, records assigned to Division 0 will be excluded from the modeling dataset.

This decision is consistent with the findings previously documented during the EDA. The original dataset will be preserved, while 753,813 records will be retained for the target divisions.

### 1.13. Filter Target Divisions

We will create a separate DataFrame containing only intervention records from operational Divisions 1 through 6.

The original combined dataset will remain unchanged. This filtered dataset will be used to calculate daily intervention counts for each division.

In [ ]:

# Keep only interventions from Divisions 1 through 6
df_model = df[df["DIVISION"].isin([1, 2, 3, 4, 5, 6])].copy()

# Display the number of retained and excluded records
print("Original records:", len(df))
print("Records for modeling:", len(df_model))
print("Excluded records:", len(df) - len(df_model))

# Verify the divisions retained
print("\nDivisions included:")
print(sorted(df_model["DIVISION"].unique()))


Original records: 754413
Records for modeling: 753813
Excluded records: 600

Divisions included:
[np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5), np.int64(6)]


### 1.14. Aggregate Daily Intervention Counts

The original dataset contains one record per published intervention. The forecasting models require daily intervention counts for each operational division.

We will group the records by calendar date and division, then count the number of interventions in each group.

The resulting dataset will contain three columns: `Date`, `DIVISION`, and `Intervention_Count`.

Each row will represent one operational division on one calendar day, consistent with the unit of analysis defined in the revised Business Problem Framing Report.

In [ ]:

# Aggregate intervention records by date and division
df_daily = (
    df_model.groupby(["Date", "DIVISION"])
    .size()
    .reset_index(name="Intervention_Count")
)

# Display the dimensions of the daily dataset
print("Daily dataset dimensions:", df_daily.shape)

# Display the first 12 observations
display(df_daily.head(12))

# Verify that total intervention counts are preserved
print("\nOriginal modeling records:", len(df_model))
print("Total aggregated interventions:", df_daily["Intervention_Count"].sum())


Daily dataset dimensions: (14748, 3)


,Date,DIVISION,Intervention_Count
0,2020-01-01,1,35
1,2020-01-01,2,67
2,2020-01-01,3,55
3,2020-01-01,4,70
4,2020-01-01,5,64
5,2020-01-01,6,69
6,2020-01-02,1,24
7,2020-01-02,2,66
8,2020-01-02,3,58
9,2020-01-02,4,54



Original modeling records: 753813
Total aggregated interventions: 753813


**Methodological Observation**

The aggregation produced 14,748 division-day observations, consistent with the revised Business Problem Framing Report and the previous EDA findings.

The total aggregated intervention count is 753,813, matching the number of records retained for operational Divisions 1 through 6. This confirms that no intervention records were lost during aggregation.

The resulting dataset establishes the daily intervention count as the forecasting target. The continuity of each division's time series will be verified separately before proceeding with modeling.

### 1.15. Verify Daily Coverage by Division

We will verify the number of observed calendar days for each operational division and identify any missing dates within the complete study period.

This check will help determine whether the daily time series are continuous, apart from the previously identified missing date of December 31, 2024.

Missing dates will remain missing rather than being automatically assigned zero interventions.

In [ ]:

# Count observed days for each division
days_by_division = df_daily.groupby("DIVISION")["Date"].nunique()

print("Observed days by division:")
print(days_by_division)

# Check missing dates for each division
print("\nMissing dates by division:")

for division in sorted(df_daily["DIVISION"].unique()):
    division_dates = df_daily.loc[
        df_daily["DIVISION"] == division, "Date"
    ]

    missing_division_dates = all_dates.difference(division_dates)

    print(f"\nDivision {division}: {len(missing_division_dates)} missing days")
    print(missing_division_dates)


Observed days by division:
DIVISION
1    2458
2    2458
3    2458
4    2458
5    2458
6    2458
Name: Date, dtype: int64

Missing dates by division:

Division 1: 1 missing days
DatetimeIndex(['2024-12-31'], dtype='datetime64[us]', freq='D')

Division 2: 1 missing days
DatetimeIndex(['2024-12-31'], dtype='datetime64[us]', freq='D')

Division 3: 1 missing days
DatetimeIndex(['2024-12-31'], dtype='datetime64[us]', freq='D')

Division 4: 1 missing days
DatetimeIndex(['2024-12-31'], dtype='datetime64[us]', freq='D')

Division 5: 1 missing days
DatetimeIndex(['2024-12-31'], dtype='datetime64[us]', freq='D')

Division 6: 1 missing days
DatetimeIndex(['2024-12-31'], dtype='datetime64[us]', freq='D')


**Methodological Observation**

All six operational divisions contain records for the same 2,458 calendar days. December 31, 2024, is the only missing date for every division, resulting in six missing division-day observations.

No additional division-specific gaps were identified. This confirms consistent temporal coverage across the six divisions.

The missing date will be explicitly represented as missing in the modeling dataset rather than assigned zero interventions. It will also be excluded from forecast accuracy calculations because the actual intervention counts are unknown.

### 1.16. Create a Complete Division-Day Calendar

We will create a complete calendar containing every date in the study period for each of the six operational divisions.

The existing daily intervention counts will be preserved. For December 31, 2024, six additional division-day observations will be created with missing intervention counts (NaN).

This ensures that every date-division combination is explicitly represented while distinguishing missing information from zero interventions. The complete calendar also provides a consistent temporal structure for developing calendar-based lag and rolling features in later modeling steps.

In [ ]:

# Create a complete list of dates and divisions
complete_calendar = pd.MultiIndex.from_product(
    [all_dates, [1, 2, 3, 4, 5, 6]],
    names=["Date", "DIVISION"]
)

# Add missing division-day combinations
df_daily_complete = (
    df_daily.set_index(["Date", "DIVISION"])
    .reindex(complete_calendar)
    .reset_index()
)

# Display dataset dimensions
print("Original daily dataset:", df_daily.shape)
print("Complete daily dataset:", df_daily_complete.shape)

# Count missing intervention values
print(
    "Missing intervention counts:",
    df_daily_complete["Intervention_Count"].isna().sum()
)

# Display the missing observations
print("\nObservations with missing counts:")
display(
    df_daily_complete[
        df_daily_complete["Intervention_Count"].isna()
    ]
)


Original daily dataset: (14748, 3)
Complete daily dataset: (14754, 3)
Missing intervention counts: 6

Observations with missing counts:


,Date,DIVISION,Intervention_Count
10956,2024-12-31,1,NaN
10957,2024-12-31,2,NaN
10958,2024-12-31,3,NaN
10959,2024-12-31,4,NaN
10960,2024-12-31,5,NaN
10961,2024-12-31,6,NaN


**Methodological Observation**

The complete division-day calendar contains 14,754 observations, compared with 14,748 observations with recorded intervention counts.

Six additional observations were introduced for December 31, 2024, one for each operational division. Their `Intervention_Count` values remain missing (`NaN`) rather than being replaced with zero.

This approach preserves calendar continuity while distinguishing unavailable intervention counts from observed counts. The missing values will be handled explicitly during feature engineering and model evaluation.

### 1.17. Verify the Complete Daily Dataset

We will perform final consistency checks on the complete division-day dataset.

These checks will verify that each date-division combination appears only once, that the original intervention counts have been preserved, and that the only missing target values correspond to the previously identified data gap.

The results will help confirm that the daily dataset is structurally ready for subsequent modeling steps.

In [ ]:

# Check for duplicate date-division combinations
duplicate_combinations = df_daily_complete.duplicated(
    subset=["Date", "DIVISION"]
).sum()

# Verify the number of missing intervention counts
missing_counts = df_daily_complete["Intervention_Count"].isna().sum()

# Verify that the original intervention total is preserved
total_interventions = df_daily_complete["Intervention_Count"].sum()

# Check for negative intervention counts
negative_counts = (df_daily_complete["Intervention_Count"] < 0).sum()

# Display verification results
print("Duplicate date-division combinations:", duplicate_combinations)
print("Missing intervention counts:", missing_counts)
print("Total recorded interventions:", int(total_interventions))
print("Negative intervention counts:", negative_counts)

# Display the final dataset dimensions
print("\nComplete daily dataset:", df_daily_complete.shape)


Duplicate date-division combinations: 0
Missing intervention counts: 6
Total recorded interventions: 753813
Negative intervention counts: 0

Complete daily dataset: (14754, 3)


**Methodological Observation**

The final consistency checks confirmed that the complete daily dataset contains 14,754 unique division-day combinations, with no duplicate combinations or negative intervention counts.

The total of 753,813 recorded interventions was preserved, and the only six missing target values correspond to December 31, 2024, one for each division.

These results confirm the structural consistency of the daily dataset. The missing observations will remain explicitly identified and will not be used as known actual intervention counts during model evaluation.

### 1.18. Save the Prepared Daily Dataset

The complete division-day dataset has passed the structural consistency checks.

We will save the prepared dataset as a CSV file to preserve the output of the data preparation stage and support its reuse in subsequent modeling steps.

In [ ]:

# Save the prepared daily dataset
output_file = "../../data/interim/modelling/SIM_Daily_Interventions.csv"

df_daily_complete.to_csv(output_file, index=False)

print("Dataset saved to:", output_file)


Dataset saved to: ../../data/interim/modelling/SIM_Daily_Interventions.csv


## 1.19. Stage 1 Conclusion — Modeling Dataset: Data Preparation

The data preparation stage has been completed. The original SIM intervention datasets were combined, validated, and transformed into daily intervention counts for operational Divisions 1 through 6.

The resulting dataset passed the structural consistency checks and preserves the original intervention counts for the target divisions.

### Final Dataset Summary

| Metric | Result |
|---|---:|
| Original intervention records | 754,413 |
| Records retained for Divisions 1–6 | 753,813 |
| Calendar days | 2,459 |
| Division-day combinations | 14,754 |
| Observations with valid target counts | 14,748 |
| Observations with missing target counts | 6 |
| Duplicate date-division combinations | 0 |
| Negative intervention counts | 0 |

### Key Methodological Decisions

- Records assigned to Division 0 were excluded because the forecasting target covers only Divisions 1 through 6.
- December 31, 2024, was identified as the only missing calendar day. Its six division-day observations were retained with missing target values (`NaN`) rather than replaced with zero.
- The original intervention counts were preserved during aggregation.
- The complete division-day calendar was created to maintain the correct temporal structure for subsequent modeling steps.

### Output

The prepared dataset was saved as:

`SIM_Daily_Interventions.csv`

Each observation represents one operational division on one calendar day, with `Intervention_Count` as the forecasting target.

### Next Stage: Baseline Forecasting

The next stage will implement and evaluate a baseline based on recent observations for the same day of the week, as defined in the revised Business Problem Framing Report.

The baseline will respect the forecasting information cutoff: forecasts for day D will be prepared on D−1 using historical intervention counts available through D−3.

**The daily dataset is structurally prepared, but it is not yet the final feature-engineered training dataset.** Predictive features, including lagged counts, rolling statistics, and calendar variables, will be developed in the Feature Engineering stage.